In [57]:
import numpy as np 

import matplotlib.pyplot as plt

In [58]:
df = pd.read_csv('mental_health_dataset.csv')
df

,text,status
0,"""My mind is a never-ending cycle of worry, and...",anxiety
1,Despite the sun shining and birds singing outs...,bipolar
2,"I'm drowning in responsibilities, each one dem...",stress
3,"""My emotions shift like the wind, leaving me u...",personality disorder
4,"I'm trapped in a whirlwind of thoughts, unable...",anxiety
...,...,...
103483,"I long to embrace my true self, yet the mirror...",personality disorder
103484,"""My mind is a whirlwind of worst-case scenario...",anxiety
103485,Bipolar disorder: From elated excitement to cr...,bipolar
103486,"""Despite my best efforts, the constant stream ...",stress


In [59]:
def find_length(text):
    return len(str(text).split())


df['length'] = df['text'].apply(find_length)

df = df[df['length']<=500]

df['length'].describe()


count    101798.000000
mean         70.728560
std          81.962127
min           1.000000
25%          24.000000
50%          34.000000
75%          97.000000
max         500.000000
Name: length, dtype: float64

In [60]:
MAX_LEN = 120 
MAX_WORDS = 30000

In [61]:
# Cleaning the text using NLP 

# Lower case conversion 
df['text'] = df['text'].str.lower()

# remove html tags 
import re 
def remove_html_tags(text):
    return re.sub('<.*?>','',text)
df['text'] = df['text'].apply(remove_html_tags)


# remove url's 
def remove_urls(text):
    return re.sub('http\S+|www\.\S+','',text)
df['text'] = df['text'].apply(remove_urls)

#remove punctuation 
import string 
exclude = string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('','',exclude))
df['text'] = df['text'].apply(remove_punctuation)


# # remove stop words 
# import nltk
# from nltk.corpus import stopwords
# def remove_stopwords(text):
#     new_text= []
#     for word in text:
#         if word in stopwords.words('english'):
#             new_text.append('')
#         else:
#             new_text.append(word)
#     return " ".join(new_text)
# df['text'] = df['text'].apply(remove_stopwords)

# #spelling correction 
# from textblob import TextBlob 
# def correct_sent(text):
#     return str(Textblob(text).correct())
# df['text'] = df['text'].apply(correct_sent)


import re 
def remove_spaces(text):
    return re.sub(r"\s+"," ",text).strip()
df['text'] = df['text'].apply(remove_spaces)
df


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19912\288920173.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].str.lower()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19912\288920173.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(remove_html_tags)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19912\288920173.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

,text,status,length
0,my mind is a neverending cycle of worry and ev...,anxiety,96
1,despite the sun shining and birds singing outs...,bipolar,46
2,im drowning in responsibilities each one deman...,stress,30
3,my emotions shift like the wind leaving me unc...,personality disorder,32
4,im trapped in a whirlwind of thoughts unable t...,anxiety,27
...,...,...,...
103483,i long to embrace my true self yet the mirrors...,personality disorder,27
103484,my mind is a whirlwind of worstcase scenarios ...,anxiety,24
103485,bipolar disorder from elated excitement to cru...,bipolar,16
103486,despite my best efforts the constant stream of...,stress,29


In [64]:
# Label Encoding 


from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder() 
df['label id'] = label_encoder.fit_transform(df['status'])

NUM_CLASSES= len(label_encoder.classes_)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_19912\2334004129.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label id'] = label_encoder.fit_transform(df['status'])


In [65]:
# Train test Split 

from sklearn.model_selection import train_test_split
X = df['text']
Y = df['label id']
X_train , X_test , Y_train, Y_test = train_test_split(
    X ,
    Y,
    random_state = 42 , 
    test_size = 0.2 ,
    stratify=Y
)

In [66]:
print(X_train.shape , X_test.shape)
print(Y_train.shape,Y_test.shape)

(81438,) (20360,)
(81438,) (20360,)


In [73]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    num_words = MAX_WORDS,
    oov_token = "<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen = MAX_LEN,
    padding = "post",
    truncating = "post"
)

X_test_pad = pad_sequences(
    X_test_seq , 
    maxlen = MAX_LEN,
    padding = "post",
    truncating = "post"
)



In [89]:
import tensorflow 
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense , Embedding , LSTM , Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Early Stopping 
callback = EarlyStopping(
    monitor = 'val_loss',
    restore_best_weights = True , 
    patience = 3
)

model = Sequential()

# Embedding layer
model.add(Embedding(
    input_dim = MAX_WORDS,
    output_dim = 128 
))

# LSTM layer 

model.add(LSTM(128 , return_sequences=False))

#Dropout layer

model.add(Dropout(0.5))

# Dense layer 

model.add(Dense(
    NUM_CLASSES , activation = 'softmax' 
))



In [90]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer = 'adam',
              metrics = ['accuracy'])

In [92]:
model.fit(X_train_pad,
          Y_train , 
          validation_split=0.1,
          epochs = 100,
          batch_size = 64,
          callbacks = [callback]
)

Epoch 1/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 261s 228ms/step - accuracy: 0.4033 - loss: 1.3436 - val_accuracy: 0.6088 - val_loss: 0.9353
Epoch 2/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 170s 148ms/step - accuracy: 0.6989 - loss: 0.6709 - val_accuracy: 0.8164 - val_loss: 0.4520
Epoch 3/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 167s 146ms/step - accuracy: 0.8551 - loss: 0.3756 - val_accuracy: 0.8639 - val_loss: 0.3788
Epoch 4/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 224s 195ms/step - accuracy: 0.8902 - loss: 0.2982 - val_accuracy: 0.8727 - val_loss: 0.3630
Epoch 5/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 231s 201ms/step - accuracy: 0.9134 - loss: 0.2423 - val_accuracy: 0.8668 - val_loss: 0.3797
Epoch 6/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 339s 296ms/step - accuracy: 0.9325 - loss: 0.1971 - val_accuracy: 0.8684 - val_loss: 0.4032
Epoch 7/100
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 225s 196ms/step - accuracy: 0.9465 - loss: 0.1586 - val_accuracy: 0.8655 - val_loss: 0.4433


In [96]:
accuracy , loss = model.evaluate(X_test_pad,Y_test)
print("Accuracy :" , accuracy , "Loss" ,loss)

637/637 ━━━━━━━━━━━━━━━━━━━━ 43s 67ms/step - accuracy: 0.8662 - loss: 0.3740
Accuracy : 0.37401312589645386 Loss 0.8662082552909851


In [103]:
# Rename for clarity
mh_model = model
mh_tokenizer = tokenizer
mh_label_encoder = label_encoder


In [104]:
import joblib
import os

os.makedirs("model/mental_health", exist_ok=True)

# Save trained model
mh_model.save("model/mental_health/mental_health_lstm_model.h5")

# Save tokenizer and label encoder
joblib.dump(mh_tokenizer, "model/mental_health/mh_tokenizer.pkl")
joblib.dump(mh_label_encoder, "model/mental_health/mh_label_encoder.pkl")


['model/mental_health/mh_label_encoder.pkl']